In [6]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [7]:
# -----------------------------------------------------------------------------
# 1. Загрузка сырых строк (без авто‑парсинга чисел)
# -----------------------------------------------------------------------------
train_raw = pd.read_csv('train.csv', sep=';', dtype=str, keep_default_na=False)
test_raw  = pd.read_csv('test.csv', sep=';', dtype=str, keep_default_na=False)

In [8]:
# -----------------------------------------------------------------------------
# 2. Универсальный парсер чисел (запятая = тысяч/десятичная, пробелы и т.д.)
# -----------------------------------------------------------------------------
def safe_parse_float(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "" or x.lower() in ("nan", "none", "null"):
        return np.nan

    x = x.replace(" ", "")
    x = x.replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return np.nan

In [9]:
# -----------------------------------------------------------------------------
# 3. Определение типов колонок (категориальные/числовые)
# -----------------------------------------------------------------------------
IGNORE_COLS = ['id', 'dt', 'target', 'w']
all_cols = [c for c in train_raw.columns if c not in IGNORE_COLS]

known_cat = [
    'gender',
    'adminarea',
    'incomeValueCategory',
    'city_smart_name',
    'pil',
    'addrref',
    'acard',

    'dp_ewb_last_employment_position',
    'dp_ewb_last_organization',

    'incomeValueCategory',

    'nonresident_flag',
    'client_active_flag',
    'accountsalary_out_flag',
    'blacklist_flag',

    'vert_has_app_ru_tinkoff_investing',
    'vert_has_app_ru_vtb_invest',
    'vert_has_app_ru_cian_main',
    'vert_has_app_ru_raiffeisennews',
]

cat_cols = [c for c in known_cat if c in train_raw.columns]

num_cols = [c for c in all_cols if c not in cat_cols]

In [10]:
# -----------------------------------------------------------------------------
# 4. Заполнение пропусков: медианы для числовых, 'MISSING' для категориальных
# -----------------------------------------------------------------------------
train_num = pd.DataFrame(index=train_raw.index)
test_num  = pd.DataFrame(index=test_raw.index)

for col in num_cols:
    train_num[col] = train_raw[col].apply(safe_parse_float)
    test_num[col]  = test_raw[col].apply(safe_parse_float)

medians = {}
for col in num_cols:
    m = train_num[col].median()
    medians[col] = m
    train_num[col].fillna(m, inplace=True)
    test_num[col].fillna(m, inplace=True)

train_cat = pd.DataFrame(index=train_raw.index)
test_cat  = pd.DataFrame(index=test_raw.index)
for col in cat_cols:
    train_cat[col] = train_raw[col].copy()
    test_cat[col]  = test_raw[col].copy()
    # Замена пропусков строкой 'MISSING' (LightGBM обработает как новую категорию)
    train_cat[col] = train_cat[col].replace('', 'MISSING').fillna('MISSING')
    test_cat[col]  = test_cat[col].replace('', 'MISSING').fillna('MISSING')

In [11]:
# -----------------------------------------------------------------------------
# 5. Временные признаки из dt
# -----------------------------------------------------------------------------
def add_date_features(df, dt_series):
    dt = pd.to_datetime(dt_series, errors="coerce")
    df['dt_year']   = dt.dt.year
    df['dt_month']  = dt.dt.month
    df['dt_quarter'] = dt.dt.quarter
    df['dt_dayofweek'] = dt.dt.dayofweek
    df['dt_is_month_start'] = dt.dt.is_month_start.astype(int)
    df['dt_is_month_end']   = dt.dt.is_month_end.astype(int)
    df["dt_dayofyear"] = dt.dt.dayofyear
    df["dt_week"] = dt.dt.isocalendar().week.astype(int)
    df["month_sin"] = np.sin(2*np.pi*df["dt_month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["dt_month"]/12)
    df["week_sin"] = np.sin(2*np.pi*df["dt_week"]/52)
    df["week_cos"] = np.cos(2*np.pi*df["dt_week"]/52)
    return df

train_num = add_date_features(train_num, train_raw['dt'])
test_num  = add_date_features(test_num,  test_raw['dt'])

In [12]:
# -----------------------------------------------------------------------------
# 6. Сборка финальных матриц
# -----------------------------------------------------------------------------
X_train = pd.concat([train_num, train_cat], axis=1)
X_test  = pd.concat([test_num,  test_cat],  axis=1)

y_train = train_raw['target'].apply(safe_parse_float).astype(float)
print(y_train.describe())
w_train = train_raw['w'].apply(safe_parse_float).astype(float)

# Убедимся, что категориальные признаки в X_train имеют тип object
for c in cat_cols:
    # объединяем train и test, чтобы категории совпадали
    all_values = pd.concat([X_train[c], X_test[c]], axis=0).astype(str)

    categories = pd.Categorical(all_values).categories

    X_train[c] = pd.Categorical(X_train[c].astype(str), categories=categories)
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=categories)
print(X_train.dtypes.value_counts())

count    7.678600e+04
mean     9.264824e+04
std      1.124090e+05
min      2.000000e+04
25%      3.970997e+04
50%      6.275413e+04
75%      1.002017e+05
max      1.500000e+06
Name: target, dtype: float64
float64     207
category      7
int32         5
int64         3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64


In [13]:
# -----------------------------------------------------------------------------
# 7. Проверка утечек (корреляция числовых признаков с target)
# -----------------------------------------------------------------------------
print("Проверка утечек (топ-10 корреляций по модулю):")
corr = X_train[num_cols].apply(lambda s: s.corr(y_train), axis=0).abs().sort_values(ascending=False)
print(corr.head(10))

Проверка утечек (топ-10 корреляций по модулю):
first_salary_income                         0.928217
salary_6to12m_avg                           0.927699
dp_payoutincomedata_payout_avg_6_month      0.672173
dp_payoutincomedata_payout_avg_3_month      0.644840
dp_payoutincomedata_payout_sum_3_month      0.644542
turn_cur_db_avg_act_v2                      0.640399
turn_cur_cr_avg_act_v2                      0.638592
dp_payoutincomedata_payout_avg_prev_year    0.630530
turn_cur_cr_avg_v2                          0.630285
turn_cur_cr_sum_v2                          0.630285
dtype: float64


In [14]:
# -----------------------------------------------------------------------------
# 8. Параметры LightGBM
# -----------------------------------------------------------------------------
params = {
    'objective': 'mae',
    'metric': 'mae',
    'learning_rate': 0.02,
    'num_leaves': 64,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 1.0,
    'lambda_l2': 5.0,
    'verbose': -1,
    'seed': 42,
    'num_threads': -1
}

In [15]:
# -----------------------------------------------------------------------------
# 9. Holdout по последнему месяцу
# -----------------------------------------------------------------------------

train_raw["_dt_parsed"] = pd.to_datetime(train_raw["dt"], dayfirst=True)

months = train_raw["_dt_parsed"].dt.to_period("M")

# Последний месяц train (июнь 2024)
last_month = months.max()

train_mask = months < last_month
valid_mask = months == last_month

X_tr = X_train.loc[train_mask]
y_tr = y_train.loc[train_mask]
w_tr = w_train.loc[train_mask]

X_val = X_train.loc[valid_mask]
y_val = y_train.loc[valid_mask]
w_val = w_train.loc[valid_mask]

print("Train months:", sorted(months[train_mask].unique()))
print("Valid month :", last_month)

dtrain = lgb.Dataset(
    X_tr,
    label=y_tr,
    weight=w_tr,
    categorical_feature=cat_cols,
)

dval = lgb.Dataset(
    X_val,
    label=y_val,
    weight=w_val,
    categorical_feature=cat_cols,
)

model = lgb.train(
    params,
    dtrain,
    valid_sets=[dval],
    num_boost_round=10000,
    callbacks=[
        lgb.early_stopping(500),
        lgb.log_evaluation(100),
    ],
)

pred = model.predict(X_val)

wmae = mean_absolute_error(
    y_val,
    pred,
    sample_weight=w_val,
)

print(f"WMAE = {wmae:.5f}")
print(f"Best iteration = {model.best_iteration}")

final_num_round = model.best_iteration

Train months: [Period('2024-01', 'M'), Period('2024-02', 'M'), Period('2024-03', 'M'), Period('2024-04', 'M'), Period('2024-05', 'M')]
Valid month : 2024-06
Training until validation scores don't improve for 500 rounds
[100]	valid_0's l1: 71193
[200]	valid_0's l1: 64991.7
[300]	valid_0's l1: 63355.5
[400]	valid_0's l1: 62608.8
[500]	valid_0's l1: 62314
[600]	valid_0's l1: 62120.3
[700]	valid_0's l1: 61955.6
[800]	valid_0's l1: 61781
[900]	valid_0's l1: 61640.4
[1000]	valid_0's l1: 61565.4
[1100]	valid_0's l1: 61538
[1200]	valid_0's l1: 61501.2
[1300]	valid_0's l1: 61479.1
[1400]	valid_0's l1: 61469.8
[1500]	valid_0's l1: 61447
[1600]	valid_0's l1: 61364.6
[1700]	valid_0's l1: 61253.3
[1800]	valid_0's l1: 61160.5
[1900]	valid_0's l1: 61065.5
[2000]	valid_0's l1: 60974.1
[2100]	valid_0's l1: 60929.2
[2200]	valid_0's l1: 60904.3
[2300]	valid_0's l1: 60901.7
[2400]	valid_0's l1: 60903.9
[2500]	valid_0's l1: 60897.7
[2600]	valid_0's l1: 60893.2
[2700]	valid_0's l1: 60889.9
[2800]	valid_0's 

In [16]:
# -----------------------------------------------------------------------------
# 10. Обучение финальной модели на всех данных
# -----------------------------------------------------------------------------
full_train = lgb.Dataset(X_train, label=y_train, weight=w_train,
                         categorical_feature=cat_cols)
final_model = lgb.train(params, full_train, num_boost_round=final_num_round)

# Важность признаков
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print("\nТоп-20 признаков:")
print(importance.head(20))


Топ-20 признаков:
                                               feature     importance
0                               turn_cur_cr_avg_act_v2  553960.718756
222                    dp_ewb_last_employment_position  437022.618798
216                                          adminarea  409431.695348
218                                    city_smart_name  391052.708675
1                                    salary_6to12m_avg  319956.233830
7                                   turn_cur_cr_avg_v2  173224.661240
2                              hdb_bki_total_max_limit  164354.256794
5                                          incomeValue  159600.700524
15                              turn_cur_db_avg_act_v2  154446.376627
14                                  turn_cur_db_sum_v2  144688.150227
23   avg_by_category__amount__sum__cashflowcategory...  131723.790481
4                           hdb_bki_total_cc_max_limit  129135.404180
9                          hdb_bki_total_pil_max_limit  110670.016849
2

In [17]:
# -----------------------------------------------------------------------------
# 11. Предсказание и сохранение
# -----------------------------------------------------------------------------
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({
    'id': test_raw['id'].astype(int),
    'predict': test_pred
})
submission.to_csv('submission.csv', index=False)
print("\nРезультат сохранён в submission.csv")


Результат сохранён в submission.csv


In [18]:
import pickle

with open("lightgbm_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Модель сохранена в lightgbm_model.pkl")

Модель сохранена в lightgbm_model.pkl
